# Prompt Routing Bandits in 3 Minutes

**The Problem:** You have 5 different LLM prompt templates for commodity analysis. Which one should you use for each request? Guessing wrong costs you money (poor analysis quality) and time (re-prompting).

**The Solution:** Let a multi-armed bandit learn which prompts work best for different task types while serving real requests. No A/B testing required.

**Applied Context:** Commodity trading desk routing prompts for extraction, analysis, signals, and scenario planning.

**Time to Working:** 3 minutes

---

## Setup: Install Dependencies

The only dependencies are numpy for sampling and matplotlib for visualization.

In [ ]:
# Install required packages
!pip install numpy matplotlib -q

## Define 5 Prompt Templates for Commodity Trading

Each prompt template has a different style and purpose. The bandit will learn which one works best for different types of tasks.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Five different prompt templates for commodity analysis
prompts = {
    "structured": "Extract key data points from this commodity report into JSON format: {report}",
    "evidence_only": "Using ONLY the provided data sources, analyze: {query}. If data is insufficient, say so.",
    "analytical": "Provide quantitative fundamental analysis of {commodity} including supply/demand balance, inventory levels, and price drivers.",
    "signal": "Generate a trading signal (long/short/neutral) for {commodity} with confidence level and supporting evidence.",
    "scenario": "Present bull/base/bear scenarios for {commodity} over the next {timeframe} with probability estimates."
}

prompt_names = list(prompts.keys())
print(f"Initialized {len(prompt_names)} prompt templates: {prompt_names}")

## Simulate True Quality Scores

In production, these would be human ratings or automated quality metrics. Here we simulate realistic performance where:
- "structured" excels at data extraction
- "signal" excels at trading decisions
- "analytical" excels at fundamental analysis
- "scenario" excels at forward-looking scenarios
- "evidence_only" is more conservative but reliable

**Key Insight:** Different prompts perform differently on different task types. The bandit must learn these patterns.

In [ ]:
# Define task types that arrive at the trading desk
task_types = ["extraction", "trading_decision", "fundamental_analysis", "scenario_planning"]

# True quality scores (0-1) for each prompt on each task type
# Rows = prompts, Columns = task types
true_quality = {
    "structured":    [0.90, 0.55, 0.65, 0.60],  # Best for extraction
    "evidence_only": [0.75, 0.70, 0.75, 0.65],  # Reliable but not exceptional
    "analytical":    [0.60, 0.65, 0.85, 0.70],  # Best for fundamental analysis
    "signal":        [0.50, 0.90, 0.70, 0.65],  # Best for trading decisions
    "scenario":      [0.55, 0.60, 0.70, 0.88],  # Best for scenario planning
}

# Convert to matrix for easier indexing
quality_matrix = np.array([true_quality[p] for p in prompt_names])

print("True Quality Matrix (rows=prompts, cols=task_types):")
print(f"Tasks: {task_types}")
for i, name in enumerate(prompt_names):
    print(f"{name:15} {quality_matrix[i]}")

## Implement Thompson Sampling Prompt Router

Thompson Sampling maintains a Beta distribution for each prompt's quality score and samples from it to make routing decisions.

**Algorithm:**
1. For each prompt, maintain success count (alpha) and failure count (beta)
2. When a request arrives, sample from each prompt's Beta(alpha, beta) distribution
3. Route to the prompt with the highest sample
4. Update counts based on observed quality

In [ ]:
class ThompsonSamplingRouter:
    def __init__(self, n_prompts):
        # Start with uniform prior: Beta(1, 1)
        self.alpha = np.ones(n_prompts)  # Success counts
        self.beta = np.ones(n_prompts)   # Failure counts
    
    def select_prompt(self):
        # Sample from each prompt's Beta distribution
        samples = np.random.beta(self.alpha, self.beta)
        return np.argmax(samples)
    
    def update(self, prompt_idx, quality_score):
        # Update Beta distribution parameters
        # quality_score is in [0, 1], treat as probability of success
        self.alpha[prompt_idx] += quality_score
        self.beta[prompt_idx] += (1 - quality_score)

print("Thompson Sampling router initialized")

## Run 500 Simulated Requests

Simulate a realistic workload where:
- Tasks arrive with different types (extraction, trading decisions, etc.)
- The router selects a prompt
- Quality is evaluated (in production, this would be human feedback or automated metrics)
- The router learns from the feedback

**Watch:** The router will initially explore all prompts, then exploit the best ones for each task type.

In [ ]:
# Initialize router
router = ThompsonSamplingRouter(n_prompts=len(prompt_names))

# Tracking for visualization
n_requests = 500
selections_over_time = []
rewards_over_time = []
task_history = []

# Simulate 500 requests
np.random.seed(42)
for request_num in range(n_requests):
    # Random task arrives
    task_idx = np.random.randint(0, len(task_types))
    task_history.append(task_idx)
    
    # Router selects a prompt
    prompt_idx = router.select_prompt()
    selections_over_time.append(prompt_idx)
    
    # Observe quality (with noise to simulate real feedback)
    true_quality_score = quality_matrix[prompt_idx, task_idx]
    observed_quality = np.clip(true_quality_score + np.random.normal(0, 0.1), 0, 1)
    rewards_over_time.append(observed_quality)
    
    # Update router
    router.update(prompt_idx, observed_quality)

print(f"Processed {n_requests} requests")
print(f"Average quality score: {np.mean(rewards_over_time):.3f}")

## Visualize Learning Behavior

Plot how prompt selection evolved over time. You should see:
1. **Early exploration** - all prompts tried roughly equally
2. **Convergence** - better prompts selected more frequently
3. **Continued exploration** - occasional tries of sub-optimal prompts (Thompson Sampling property)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# Plot 1: Prompt selection frequency over time (rolling window)
window = 50
for prompt_idx, prompt_name in enumerate(prompt_names):
    selections = np.array(selections_over_time) == prompt_idx
    rolling_freq = np.convolve(selections, np.ones(window)/window, mode='valid')
    axes[0].plot(rolling_freq, label=prompt_name, linewidth=2)

axes[0].set_xlabel('Request Number', fontsize=12)
axes[0].set_ylabel('Selection Frequency (50-request window)', fontsize=12)
axes[0].set_title('Prompt Router Learning: Selection Frequency Over Time', fontsize=14, fontweight='bold')
axes[0].legend(loc='upper right')
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim([0, 1])

# Plot 2: Cumulative average quality score
cumulative_avg_quality = np.cumsum(rewards_over_time) / np.arange(1, n_requests + 1)
axes[1].plot(cumulative_avg_quality, linewidth=2, color='green', label='Bandit Routing')
axes[1].axhline(y=np.mean(rewards_over_time), color='red', linestyle='--', 
                label=f'Final Average: {np.mean(rewards_over_time):.3f}')
axes[1].set_xlabel('Request Number', fontsize=12)
axes[1].set_ylabel('Cumulative Average Quality', fontsize=12)
axes[1].set_title('Quality Improvement Over Time', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nObserve: Router learns to prefer better prompts while maintaining exploration")

## Performance Summary

Break down which prompts dominated for which task types. This shows the router learned the true quality patterns.

In [ ]:
# Analyze prompt selection by task type (last 200 requests after learning)
learning_period = 300
task_prompt_matrix = np.zeros((len(task_types), len(prompt_names)))

for i in range(learning_period, n_requests):
    task_idx = task_history[i]
    prompt_idx = selections_over_time[i]
    task_prompt_matrix[task_idx, prompt_idx] += 1

# Normalize to percentages
task_prompt_matrix = task_prompt_matrix / task_prompt_matrix.sum(axis=1, keepdims=True) * 100

print("Prompt Selection by Task Type (last 200 requests):")
print("=" * 80)
for task_idx, task_name in enumerate(task_types):
    print(f"\n{task_name.upper()}:")
    for prompt_idx, prompt_name in enumerate(prompt_names):
        percentage = task_prompt_matrix[task_idx, prompt_idx]
        if percentage > 10:  # Only show prompts selected >10% of the time
            print(f"  {prompt_name:15} {percentage:5.1f}% (true quality: {quality_matrix[prompt_idx, task_idx]:.2f})")

# Overall statistics
print("\n" + "=" * 80)
print(f"Total requests processed: {n_requests}")
print(f"Average quality score: {np.mean(rewards_over_time):.3f}")
print(f"Quality score std dev: {np.std(rewards_over_time):.3f}")

# Compare to random routing
random_quality_scores = []
for task_idx in task_history:
    random_prompt_idx = np.random.randint(0, len(prompt_names))
    random_quality_scores.append(quality_matrix[random_prompt_idx, task_idx])

print(f"\nRandom routing baseline: {np.mean(random_quality_scores):.3f}")
improvement = (np.mean(rewards_over_time) - np.mean(random_quality_scores)) / np.mean(random_quality_scores) * 100
print(f"Bandit improvement: {improvement:.1f}% better than random")

## Modify This: Experimentation Playground

Try these modifications to build intuition:

### 1. Change Quality Scores
```python
# Make one prompt universally better
true_quality["evidence_only"] = [0.95, 0.95, 0.95, 0.95]
# What happens? Does the bandit quickly converge to this prompt?
```

### 2. Add a New Prompt
```python
prompts["risk_focused"] = "Assess downside risks and tail events for {commodity} position."
# Add a row to true_quality with its performance profile
```

### 3. Change Task Distribution
```python
# Simulate a trading desk that mostly needs signals
task_weights = [0.1, 0.6, 0.2, 0.1]  # More trading decisions
task_idx = np.random.choice(len(task_types), p=task_weights)
# Does the bandit adapt to this distribution?
```

### 4. Add More Noise
```python
# Simulate inconsistent quality feedback
observed_quality = np.clip(true_quality_score + np.random.normal(0, 0.3), 0, 1)
# Does learning take longer? Does it still converge?
```

### 5. Compare Bandit Algorithms
Try implementing:
- Epsilon-greedy (10% random exploration)
- UCB (Upper Confidence Bound)
- Softmax selection

Which converges fastest? Which has the best final performance?

## What's Next?

This quick-start showed prompt routing with context-free bandits. In production, you'll want:

### 1. Contextual Bandits (Module 8)
- Use features (user role, commodity type, market conditions) to personalize routing
- Linear models or neural networks for quality prediction
- LinUCB, Thompson Sampling with linear models

### 2. Reward Design
- Combine quality metrics: accuracy, latency, cost, user satisfaction
- Delayed rewards: quality feedback arrives later
- Multi-objective optimization: maximize quality while minimizing API cost

### 3. Production Deployment
- A/B testing bandits vs. fixed routing
- Monitoring: track quality, exploration rate, convergence
- Safe exploration: constrain to high-quality prompts during critical periods
- Prompt versioning: handle prompt updates without losing learning

### 4. Advanced Techniques
- Ensemble routing: combine multiple prompts
- Cascade routing: fast prompt first, fallback to careful prompt if uncertain
- Budget allocation: balance quality vs. API cost across request types

### Real-World Resources
- **Vowpal Wabbit**: Production contextual bandit library
- **Azure Personalizer**: Managed contextual bandit service
- **OpenAI Fine-Tuning**: Alternative to routing (but less flexible)

### Copy-Paste Production Template
See `templates/prompt_router_template.py` for production-ready code with:
- Async prompt execution
- Quality metric tracking
- A/B testing framework
- Cost-aware routing

---

**Key Insight:** Prompt routing eliminates the "bad prompt tax" by learning which prompts work best. This is cheaper than fine-tuning and more flexible than fixed routing rules.